# 1. Black-Scholes prices and implied volatility

**Level:** L1. **Tags:** `options`, `black-scholes`, `implied-vol`.
**Library:** `eq_options.black_scholes` (`python/equity/01-options-pricing`).

**Plain idea.** Under Black-Scholes a European call is worth the discounted
expected payoff when the stock grows at the risk-free rate with constant
volatility. Implied volatility runs the formula backwards: the one sigma that
reproduces a quoted price.

**Formal statement** (Hull, *Options, Futures and Other Derivatives*, 11th
ed., the Black-Scholes-Merton chapter):
$C = S e^{-qT} N(d_1) - K e^{-rT} N(d_2)$, with
$d_{1,2} = [\ln(S/K) + (r - q \pm \sigma^2/2)T] / (\sigma\sqrt{T})$.
Vega is $\partial C/\partial\sigma = S e^{-qT} \varphi(d_1)\sqrt{T}$, per
unit of volatility (so per 1.00, not per 1%).

In [ ]:
import _paths  # noqa: F401  (puts the library on sys.path)

In [ ]:
import math

import numpy as np
from eq_options.black_scholes import bs_price, implied_vol
from scipy.stats import norm

## Task 1: price the textbook at-the-money call

Use `bs_price` with $S = K = 100$, $T = 1$, $r = 5\%$, $\sigma = 20\%$, no
dividend. Store it as `call`, and the matching put as `put`.

In [ ]:
S, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.20
call = bs_price(S, K, T, r, sigma, option_type="call")
put = bs_price(S, K, T, r, sigma, option_type="put")
print(f"call {call:.4f}  put {put:.4f}")

**Independent check.** The closed form written out by hand with `scipy`, and
put-call parity $C - P = S e^{-qT} - K e^{-rT}$, which holds for any model.
The value to four places is 10.4506; the checks recompute it rather than
trust a printed figure.

In [ ]:
d1 = (math.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
d2 = d1 - sigma * math.sqrt(T)
by_hand = S * norm.cdf(d1) - K * math.exp(-r * T) * norm.cdf(d2)
assert abs(call - by_hand) < 1e-12
assert abs((call - put) - (S - K * math.exp(-r * T))) < 1e-12
assert round(call, 4) == 10.4506

## Task 2: vega, two ways

Compute `vega_fd`, a central finite difference of `bs_price` in sigma with a
bump of `1e-4`. Then `vega_cf`, the closed form. Note the size: about 37.5
per unit vol, so a one-point move (0.01) is worth about 0.375. A vega that is
bigger than the price is normal for an at-the-money option; it is quoted per
unit of sigma.

In [ ]:
h = 1e-4
vega_fd = (bs_price(S, K, T, r, sigma + h) - bs_price(S, K, T, r, sigma - h)) / (2 * h)
vega_cf = S * norm.pdf(d1) * math.sqrt(T)
print(f"vega fd {vega_fd:.6f}  closed form {vega_cf:.6f}  per vol point {vega_cf / 100:.4f}")

In [ ]:
assert abs(vega_fd - vega_cf) < 1e-6
assert 37.0 < vega_cf < 38.0

## Task 3: implied volatility round trip

Build a small smile: strikes 80 to 120 with a made-up skew
$\sigma(K) = 0.20 - 0.10\ln(K/100)$. Price each strike, then recover sigma
with `implied_vol`. Store the recovered values in `iv`.

In [ ]:
strikes = np.arange(80.0, 121.0, 5.0)
true_vol = 0.20 - 0.10 * np.log(strikes / 100.0)
prices = [bs_price(S, k, T, r, v) for k, v in zip(strikes, true_vol, strict=True)]
iv = np.array([implied_vol(p, S, k, T, r) for p, k in zip(prices, strikes, strict=True)])
for k, v in zip(strikes, iv, strict=True):
    print(f"K={k:5.0f}  iv={v:.6f}")

In [ ]:
assert np.max(np.abs(iv - true_vol)) < 1e-8

**Takeaways.** (1) Parity is the cheapest test of any pricer. (2) Quote vega
with its unit. (3) An inverter is only trusted after a round trip across
strikes, including the wings.